In [ ]:
import os, math
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, mixed_precision
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score,
)

print(f"TensorFlow {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"Mixed precision support: {len(gpus) > 0}")


In [ ]:
# Mixed float16 speeds up GPU training ~2x with negligible accuracy loss.
# Comment out if you see NaN losses (rare with AdamW + label smoothing).
mixed_precision.set_global_policy("mixed_float16")
print("Global precision policy:", mixed_precision.global_policy().name)


In [ ]:
# ── UPDATE THIS ──────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/datasets/satyamsingh0912/fish-classifier-dataset/final_dataset"
# ─────────────────────────────────────────────────────────────────────────

IMG_SIZE   = (224, 224)
BATCH_SIZE = 64       # use 32 if GPU < 8 GB
SEED       = 42
AUTOTUNE   = tf.data.AUTOTUNE


In [ ]:
train_dir = Path(DATA_DIR) / "train"
class_names = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])

# Count files per class (handles jpg, jpeg, png, webp)
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
class_counts = {
    name: sum(1 for f in (train_dir / name).iterdir() if f.suffix.lower() in EXTS)
    for name in class_names
}
total_train = sum(class_counts.values())

print("=" * 65)
print("CLASS MAPPING (model class index = alphabetical folder order)")
for idx, name in enumerate(class_names):
    print(f"  class {idx}  →  folder '{name}'  →  {class_counts[name]:,} training images")
print()
print(">>> In backend/main.py set:  CLASS_NAMES =", class_names)
print("=" * 65)

# Balanced class weights: total / (n_classes * n_per_class)
# Higher weight = model penalised more for misclassifying that class
n_classes = len(class_names)
class_weight = {
    idx: total_train / (n_classes * count)
    for idx, (name, count) in enumerate(class_counts.items())
}
print()
print("Computed class weights (sklearn 'balanced' formula):")
for idx, w in class_weight.items():
    print(f"  class {idx} ({class_names[idx]}): {w:.4f}")


In [ ]:
# All augmentations operate on raw [0, 255] pixel values.
# The EfficientNetV2S backbone handles normalisation internally
# (include_preprocessing=True divides by 255 inside the model).
#
# Strategy: train on RAW fish images with heavy augmentation so the model
# learns fish-specific anatomy (eyes, skin, gills) rather than background
# cues.  This avoids the 16-hr per-image preprocessing step while still
# producing a model that is robust to background variation at inference time.

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.30),             # up to ±108 degrees
    layers.RandomZoom((-0.30, 0.05)),        # mostly zoom-in (crop effect)
    layers.RandomTranslation(0.10, 0.10),   # shift up to 10 %
    layers.RandomBrightness(0.25),           # ±25 % brightness
    layers.RandomContrast(0.30),             # contrast jitter
], name="augmentation")

# Preview to confirm augmentation looks healthy
sample_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir), image_size=IMG_SIZE, batch_size=8, seed=SEED, shuffle=True
)
for sample_imgs, _ in sample_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        aug_img = augmentation(sample_imgs[i : i + 1], training=True)[0]
        ax.imshow(aug_img.numpy().clip(0, 255).astype("uint8"))
        ax.axis("off")
    plt.suptitle("Augmentation preview — raw fish images", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
def load_split(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR + f"/{split}",
        image_size=IMG_SIZE,
        batch_size=None,          # batch AFTER augmentation
        label_mode="categorical", # one-hot labels
        shuffle=shuffle,
        seed=SEED,
    )

train_ds_raw = load_split("train", shuffle=True)
val_ds_raw   = load_split("val",   shuffle=False)
test_ds_raw  = load_split("test",  shuffle=False)

# Apply augmentation only on training data
train_ds = (
    train_ds_raw
    .map(lambda x, y: (augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds  = val_ds_raw.batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_ds_raw.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Train batches :", len(train_ds))
print("Val   batches :", len(val_ds))
print("Test  batches :", len(test_ds))


In [ ]:
def build_model(n_classes):
    # ── Backbone ──────────────────────────────────────────────────────────
    # include_preprocessing=True  →  model expects raw [0, 255] pixels and
    # applies Rescaling(1/255) internally.  Never normalise outside the model.
    base = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        include_preprocessing=True,
    )
    base.trainable = False  # Phase 1: frozen

    # ── Head ──────────────────────────────────────────────────────────────
    inputs  = tf.keras.Input(shape=(224, 224, 3), name="image_input")
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D(name="gap")(x)
    x       = layers.BatchNormalization(name="bn_head")(x)

    x       = layers.Dense(512, kernel_regularizer=regularizers.l2(1e-4),
                           name="dense_512")(x)
    x       = layers.Activation("gelu", name="gelu_1")(x)
    x       = layers.Dropout(0.40, name="drop_1")(x)

    x       = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4),
                           name="dense_256")(x)
    x       = layers.Activation("gelu", name="gelu_2")(x)
    x       = layers.Dropout(0.30, name="drop_2")(x)

    # float32 output layer for mixed-precision numerical stability
    logits  = layers.Dense(n_classes, name="logits", dtype="float32")(x)
    outputs = layers.Softmax(name="predictions", dtype="float32")(logits)

    return tf.keras.Model(inputs, outputs, name="FreshlyFishy_v2")


model = build_model(n_classes=len(class_names))
model.summary(line_length=110, expand_nested=False)


In [ ]:
# Phase 1: train only the head — backbone stays frozen.
# Monitor val_auc (more robust than val_loss for imbalanced data).
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.05,   # prevents overconfidence, improves calibration
    ),
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
    ],
)

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=8, restore_best_weights=True, mode="max",
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc", factor=0.4, patience=3, min_lr=1e-7, mode="max",
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_phase1.keras", monitor="val_auc", save_best_only=True, mode="max",
    ),
]


In [ ]:
print("=" * 65)
print("PHASE 1 — Head training with frozen EfficientNetV2S backbone")
print("=" * 65)

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weight,   # compensate for class imbalance
    callbacks=callbacks_p1,
)


In [ ]:
def plot_history(history, title=""):
    h = history.history
    # Determine which metrics are available
    metrics = [
        ("loss",     "val_loss",     "Loss"),
        ("accuracy", "val_accuracy", "Accuracy"),
        ("auc",      "val_auc",      "ROC-AUC"),
    ]
    metrics = [(t, v, l) for t, v, l in metrics if t in h]

    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
    if len(metrics) == 1:
        axes = [axes]
    for ax, (train_m, val_m, label) in zip(axes, metrics):
        ax.plot(h[train_m], label="Train", linewidth=2)
        ax.plot(h[val_m],   label="Val",   linewidth=2, linestyle="--")
        ax.set_title(label)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

plot_history(history_p1, "Phase 1 — Head Training")


In [ ]:
print("=" * 65)
print("PHASE 2 — Fine-tuning top 60 layers of EfficientNetV2S")
print("=" * 65)

base_model = model.get_layer("efficientnetv2-s")
base_model.trainable = True

# Freeze everything except the last 60 layers
for layer in base_model.layers[:-60]:
    layer.trainable = False

trainable_params = sum(tf.size(v).numpy() for v in model.trainable_variables)
total_params     = sum(tf.size(v).numpy() for v in model.variables)
print(f"Trainable: {trainable_params:,} / {total_params:,} parameters")

# Cosine decay: starts at 5e-5, decays to near-zero over 20 epochs
steps_per_epoch = math.ceil(total_train / BATCH_SIZE)
cosine_decay = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-5,
    decay_steps=20 * steps_per_epoch,
    alpha=1e-7,
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=cosine_decay, weight_decay=1e-5
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
    ],
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=10, restore_best_weights=True, mode="max",
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_phase2.keras", monitor="val_auc", save_best_only=True, mode="max",
    ),
]

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weight,
    callbacks=callbacks_p2,
)


In [ ]:
plot_history(history_p2, "Phase 2 — Fine-Tuning")


In [ ]:
print("=" * 65)
print("EVALUATION ON HELD-OUT TEST SET")
print("=" * 65)

y_true_all, y_prob_all = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true_all.extend(np.argmax(labels.numpy(), axis=1))
    y_prob_all.extend(preds)

y_true = np.array(y_true_all)
y_prob = np.array(y_prob_all)
y_pred = np.argmax(y_prob, axis=1)

# ── Metrics ──────────────────────────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

auc_score = roc_auc_score(y_true, y_prob[:, 1])
f1_w      = f1_score(y_true, y_pred, average="weighted")
f1_each   = f1_score(y_true, y_pred, average=None)
print(f"ROC-AUC           : {auc_score:.4f}")
print(f"Weighted F1       : {f1_w:.4f}")
for i, (name, f1v) in enumerate(zip(class_names, f1_each)):
    print(f"  F1 ({name:>9}): {f1v:.4f}")

# ── Bias check (per-class recall) ────────────────────────────────────────
print("\nPer-class recall (bias check — both should be similar):")
for i, name in enumerate(class_names):
    mask = y_true == i
    recall = (y_pred[mask] == i).mean()
    print(f"  {name}: {recall:.2%}  ({mask.sum()} test samples)")

# ── Confusion matrices ───────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_ylabel("True"); axes[0].set_xlabel("Predicted")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Confusion Matrix (per-class recall)")
axes[1].set_ylabel("True"); axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.show()


In [ ]:
# Verify the model attends to fish anatomy (eyes, gills, skin texture)
# rather than background — if GradCAM highlights background, augmentation
# needs to be made stronger or more training epochs are needed.

def gradcam_heatmap(model, img_array):
    """Background-suppressed GradCAM for a (1,H,W,3) image array."""
    base    = model.get_layer("efficientnetv2-s")
    # 'top_conv' is the last Conv2D in EfficientNetV2S
    try:
        last_conv_out = base.get_layer("top_conv").output
    except ValueError:
        # fallback: use the last Conv2D layer found
        for layer in reversed(base.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_out = layer.output
                break

    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[last_conv_out, model.output],
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(tf.cast(img_array, tf.float32))
        pred_idx  = tf.argmax(preds[0])
        class_score = preds[:, pred_idx]

    grads        = tape.gradient(class_score, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap      = (conv_out[0] @ pooled_grads[..., tf.newaxis]).numpy().squeeze()
    heatmap      = np.maximum(heatmap, 0)
    if heatmap.max() > 1e-8:
        heatmap /= heatmap.max()
    return heatmap, int(pred_idx), preds[0].numpy()


n_show   = 6
fig, axes = plt.subplots(n_show, 3, figsize=(12, n_show * 4))
img_iter  = iter(test_ds_raw)
shown     = 0

for img_tensor, label_tensor in img_iter:
    if shown >= n_show:
        break
    img_np    = img_tensor.numpy().astype("uint8")
    img_array = img_tensor.numpy()[np.newaxis]

    heatmap, pred_idx, probs = gradcam_heatmap(model, img_array)
    true_idx = int(np.argmax(label_tensor.numpy()))

    h, w = img_np.shape[:2]
    hm_resized = cv2.resize(heatmap, (w, h))
    hm_u8      = np.uint8(255 * hm_resized)
    hm_color   = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)
    overlay    = cv2.addWeighted(img_np, 0.60, hm_color, 0.40, 0)

    correct = pred_idx == true_idx
    colour  = "green" if correct else "red"

    axes[shown, 0].imshow(img_np)
    axes[shown, 0].set_title(f"True: {class_names[true_idx]}", fontsize=10)
    axes[shown, 0].axis("off")

    axes[shown, 1].imshow(hm_resized, cmap="jet")
    axes[shown, 1].set_title("Attention heatmap", fontsize=10)
    axes[shown, 1].axis("off")

    axes[shown, 2].imshow(overlay)
    axes[shown, 2].set_title(
        f"Pred: {class_names[pred_idx]} ({probs[pred_idx]*100:.1f}%)",
        fontsize=10, color=colour,
    )
    axes[shown, 2].axis("off")
    shown += 1

plt.suptitle(
    "GradCAM — verify model focuses on fish anatomy, not background",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.show()


In [ ]:
model.save("/kaggle/working/fish_classifier.keras")
print("Saved  →  /kaggle/working/fish_classifier.keras")

print()
print("=" * 65)
print("REQUIRED CHANGES IN backend/main.py after downloading the model")
print("=" * 65)
print()
print(f"1.  CLASS_NAMES = {class_names!r}")
print(f"    (class 0 = '{class_names[0]}', class 1 = '{class_names[1]}')")
print()
print("2.  In preprocess_roi(), change the last normalisation line to:")
print("      normalised = resized.astype(np.float32)   # NO / 255.0 !")
print("    The model's include_preprocessing=True handles it internally.")
print()
print("3.  No other changes needed — classify() already supports softmax.")
print("=" * 65)
